In [1]:
import pandas as pd

In [6]:
df=pd.read_csv("spotify_millsongdata.csv")
df

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...
...,...,...,...,...
57645,Ziggy Marley,Good Old Days,/z/ziggy+marley/good+old+days_10198588.html,Irie days come on play \r\nLet the angels fly...
57646,Ziggy Marley,Hand To Mouth,/z/ziggy+marley/hand+to+mouth_20531167.html,Power to the workers \r\nMore power \r\nPowe...
57647,Zwan,Come With Me,/z/zwan/come+with+me_20148981.html,all you need \r\nis something i'll believe \...
57648,Zwan,Desire,/z/zwan/desire_20148986.html,northern star \r\nam i frightened \r\nwhere ...


In [7]:
df.shape

(57650, 4)

In [8]:
df.isnull().sum()

artist    0
song      0
link      0
text      0
dtype: int64

In [9]:
df =df.sample(5000).drop('link', axis=1).reset_index(drop=True)

In [10]:
df.head()

,artist,song,text
0,Bonnie Raitt,Steal Your Heart Away,I know you got a different picture in mind \r...
1,Ocean Colour Scene,Song Of A Baker - Originally By The Small Faces,There's wheat in the field \r\nAnd water in t...
2,Gary Numan,We Need It,We are not \r\nWhat you've been waiting for ...
3,Tom Waits,I Hope That I Don't Fall In Love With You,Well I hope that I don't fall in love with you...
4,Hillsong,No reason to hide,We've got nothing to lose \r\nWe've got no re...


In [11]:
df['text'][0]

"I know you got a different picture in mind  \r\nSome dream world of a different kind  \r\nI know this rhythm we've been moving to is way too slow  \r\nEach time our bodies aresupposed to meet  \r\nI'm here still missing a beat  \r\nAnd when I look at you I see in your eyes  \r\nWe're still dancing to a different tune  \r\n  \r\nTake me to your dreaming place  \r\nOpen up my eyes  \r\nLet me see what it takes to be  \r\nYour lover in disguise  \r\nshow me how I do those things  \r\nThat make you want to stay  \r\n'Cause I'll Go crazy if I don't find  \r\nThe words I gotta say  \r\nTo steal your heart away  \r\n  \r\nI'll buy you diamonds, buy you golden rings  \r\nAll kinds of Beautiful things  \r\nBut that don't matter to a lover with a Restless heart  \r\nCause you still want to wake up to a Different scene  \r\nSome screen gem like the one in your dream  \r\nAnd leave me standing like a foolin' the wings  \r\nStill waiting for a part to Play  \r\n  \r\nTake me to your dreaming place

In [ ]:
# df=df.sample(5000)

In [12]:
df.shape

(5000, 3)

# Text Preprocessing

In [13]:
df['text'] = df['text'].str.lower().replace(r'^\w\s', ' ').replace(r'\n', ' ', regex = True)

In [14]:
import nltk
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()

def tokenization(txt):
    tokens = nltk.word_tokenize(txt)
    stemming = [stemmer.stem(w) for w in tokens]
    return " ".join(stemming)

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
tfid = TfidfVectorizer(analyzer='word', stop_words = 'english')
matrix = tfid.fit_transform(df['text'])
similar = cosine_similarity(matrix)

In [17]:
similar[0]

array([1.        , 0.02097259, 0.06300279, ..., 0.03156153, 0.03840336,
       0.01261689])

In [18]:
df[df['song']=='Waiting For The Man']
df['song']

0                                 Steal Your Heart Away
1       Song Of A Baker - Originally By The Small Faces
2                                            We Need It
3             I Hope That I Don't Fall In Love With You
4                                     No reason to hide
                             ...                       
4995                                          Heartland
4996                                     Big Brown Eyes
4997                                   Move To The City
4998                                         Used To Be
4999                                Here Comes The Moon
Name: song, Length: 5000, dtype: object

# Recommendation Function

In [19]:
def recommendation(song_df):
    idx = df[df['song'] == song_df].index[0]
    distances = sorted(list(enumerate(similar[idx])),reverse=True,key=lambda x:x[1])
    
    songs = []
    for m_id in distances[1:5]:
        songs.append(df.iloc[m_id[0]].song)  
    return songs

In [20]:
recommendation("Happy")

['Inside Out', 'New Frontier', 'I Shine, You Shine', "I'll Be There For You"]

In [21]:
import pickle
pickle.dump(similar, open("similarity.pkl","wb"))
pickle.dump(df,open("df.pkl","wb"))